# Tutorial 4: Causal Traces, Completeness & Evaluation

**ARIA: Causal-Aware Reasoning for Materials Discovery**

---

## Introduction

This tutorial covers the evaluation and transparency layer of ARIA. We examine:

1. **Causal traces** in `ARIAResult` -- how ARIA explains its reasoning
2. **Causal completeness scoring** -- when is evidence sufficient?
3. **Contextual tunneling** -- how incomplete evidence can hurt reasoning
4. **Benchmark evaluation** with `BenchmarkRunner`
5. **Evaluation metrics** (causal coherence, source grounding, PSP consistency)
6. **LLM-as-judge** evaluation with `LLMJudge`

ARIA's central thesis is: **retrieval without causal completeness can harm reasoning.** This tutorial demonstrates why, and how ARIA's evaluation framework quantifies the difference.

---

## 1. Setup

In [ ]:
import networkx as nx
import json

from aria.types import (
    ARIAResult, CausalTraceStep, EngineMode, ReasoningTier, PSPType,
    ChainOfThought, KnowledgeSource, ReasoningStep,
)
from aria.kg.schema import classify_node_layer, psp_layers_covered, classify_path_layers
from aria.retrieval.path_search import find_psp_paths, extract_mechanisms
from aria.retrieval.completeness import (
    causal_completeness_score,
    per_path_completeness,
    identify_missing_layers,
    infer_required_layers,
    PSPLayer,
)
from aria.retrieval.evidence_ranker import rank_paths_by_evidence, path_score_details
from aria.visualization.trace_viz import plot_causal_trace, plot_tier_comparison

In [ ]:
# Build the demo KG
kg = nx.DiGraph()

kg.add_edge(
    "CVD temperature 750C", "improved crystallinity",
    mechanism="High CVD temperature promotes ordered MoS2 crystal growth",
    affected_property="crystallinity", confidence=0.9,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "CVD temperature 750C", "reduced defect density",
    mechanism="Elevated temperature anneals sulfur vacancies",
    affected_property="defect density", confidence=0.85,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="decreases",
)
kg.add_edge(
    "doping concentration Nb", "doping_level",
    mechanism="Nb substitution at Mo sites introduces carriers",
    affected_property="doping_level", confidence=0.92,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "CVD temperature 750C", "grain growth",
    mechanism="Higher temperature increases grain size",
    affected_property="grain_size", confidence=0.8,
    psp_type=PSPType.PROCESSING_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "grain growth", "improved crystallinity",
    mechanism="Larger grains reduce grain boundary scattering",
    affected_property="crystallinity", confidence=0.75,
    psp_type=PSPType.STRUCTURE_TO_STRUCTURE.value, relation="increases",
)
kg.add_edge(
    "improved crystallinity", "higher carrier mobility",
    mechanism="Ordered crystal lattice enables efficient carrier transport",
    affected_property="carrier mobility", confidence=0.88,
    psp_type=PSPType.STRUCTURE_TO_PROPERTY.value, relation="increases",
)
kg.add_edge(
    "reduced defect density", "higher carrier mobility",
    mechanism="Fewer scattering centres boost mobility",
    affected_property="carrier mobility", confidence=0.82,
    psp_type=PSPType.STRUCTURE_TO_PROPERTY.value, relation="increases",
)

print(f"Demo KG: {kg.number_of_nodes()} nodes, {kg.number_of_edges()} edges")

## 2. Understanding ARIAResult.causal_trace

The `causal_trace` field in `ARIAResult` provides a step-by-step PSP chain showing how the prediction was derived. Each `CausalTraceStep` contains:

- `processing`: The synthesis condition that initiated this causal step
- `structure`: The structural change caused by the processing
- `property_`: The resulting property change
- `evidence_text`: Supporting evidence from the KG
- `evidence_doi`: DOI of the source paper (if available)
- `confidence`: Edge confidence from the KG

In [ ]:
# Create an ARIAResult with a detailed causal trace
result_with_trace = ARIAResult(
    answer={"carrier_type": "n-type", "mobility": "50 cm2/Vs"},
    tier=ReasoningTier.DIRECT,
    confidence=0.85,
    reasoning_type="direct_path",
    causal_trace=[
        CausalTraceStep(
            processing="CVD temperature 750C",
            structure="improved crystallinity",
            property_="higher carrier mobility",
            evidence_text="High CVD temperature promotes ordered crystal growth",
            evidence_doi="10.1234/mos2-cvd",
            confidence=0.9,
        ),
        CausalTraceStep(
            processing="CVD temperature 750C",
            structure="reduced defect density",
            property_="higher carrier mobility",
            evidence_text="Elevated temperature anneals sulfur vacancies",
            evidence_doi="10.1234/mos2-cvd",
            confidence=0.85,
        ),
    ],
    missing_evidence=[],
    kg_paths_used=2,
    kg_paths=[
        "CVD temperature 750C -> improved crystallinity -> higher carrier mobility",
        "CVD temperature 750C -> reduced defect density -> higher carrier mobility",
    ],
    mode="aria",
    model="qwen2:7b",
    latency_ms=1234.5,
)

print("Causal Trace Analysis:")
print("=" * 60)
for i, step in enumerate(result_with_trace.causal_trace, 1):
    print(f"\nTrace Step {i}:")
    print(f"  Processing:  {step.processing}")
    print(f"  Structure:   {step.structure}")
    print(f"  Property:    {step.property_}")
    print(f"  Evidence:   {step.evidence_text}")
    print(f"  DOI:        {step.evidence_doi}")
    print(f"  Confidence: {step.confidence}")
    
    # Classify each element
    p_layer = classify_node_layer(step.processing)
    s_layer = classify_node_layer(step.structure)
    pr_layer = classify_node_layer(step.property_)
    print(f"  PSP layers:  {p_layer} -> {s_layer} -> {pr_layer}")

## 3. Causal Completeness Scoring

ARIA's causal completeness metric quantifies whether retrieved evidence covers all the PSP layers required by a query:

$$C(E, q) = \frac{|L(E) \cap L_{\text{req}}(q)|}{|L_{\text{req}}(q)|}$$

A completeness score of **1.0** means all required layers are covered. A score below 1.0 means some layers are missing, and the prediction is at risk of **contextual tunneling** -- where partial evidence creates a false sense of support.

In [ ]:
# Compute completeness for different types of queries
queries_and_paths = [
    {
        "query": "Predict carrier mobility for CVD MoS2 at 750C",
        "description": "Full forward prediction (needs P, S, Pr)",
        "keywords_start": ["temperature", "CVD"],
        "keywords_end": ["mobility"],
    },
    {
        "query": "How does temperature affect crystallinity?",
        "description": "Processing-Structure query (needs P, S)",
        "keywords_start": ["temperature"],
        "keywords_end": ["crystallinity"],
    },
    {
        "query": "What is the carrier mobility of MoS2?",
        "description": "Property-only query (needs Pr)",
        "keywords_start": ["mobility", "carrier"],
        "keywords_end": ["mobility", "carrier"],
    },
]

print("Causal Completeness Analysis:")
print("=" * 70)

for item in queries_and_paths:
    paths = find_psp_paths(
        kg,
        start_keywords=item["keywords_start"],
        end_keywords=item["keywords_end"],
        max_hops=4,
    )
    
    # Compute completeness
    score = causal_completeness_score(kg, paths, item["query"]) if paths else 0.0
    missing = identify_missing_layers(kg, paths, item["query"]) if paths else set()
    required = infer_required_layers(item["query"])
    
    print(f"\nQuery: {item['query']}")
    print(f"  Description:    {item['description']}")
    print(f"  Paths found:    {len(paths)}")
    print(f"  Required layers: {[l.value for l in required]}")
    print(f"  Completeness:   {score:.2f}")
    print(f"  Missing layers: {[l.value for l in missing]}")
    
    # Per-path analysis
    if paths:
        per_path = per_path_completeness(kg, paths, item["query"])
        for j, (path, p_score) in enumerate(per_path[:3], 1):
            layers = psp_layers_covered(path, kg)
            print(f"  Path {j}: score={p_score:.2f}, layers={layers}")
            print(f"           {' -> '.join(path)}")

## 4. Contextual Tunneling: How Incomplete Evidence Hurts

**Contextual tunneling** occurs when partial evidence creates a narrow context that misleads the LLM. Consider two scenarios:

1. **No KG evidence** (Tier 3): The LLM draws only on parametric knowledge, which may be vague but not misleading.

2. **Incomplete KG evidence** (naive retrieval): The LLM sees relevant-sounding edges (e.g., Processing -> Property shortcuts) but lacks the structural explanation that would justify the causal claim. This can lead to **confident but incorrect** predictions.

ARIA addresses this by **gating evidence activation**: if the retrieved evidence is causally incomplete, ARIA does not activate it at Tier 1. Instead, it falls to Tier 2 (analogical) or Tier 3 (fallback), avoiding the contextual tunneling problem.

In [ ]:
# Demonstrate contextual tunneling
print("Contextual Tunneling Demonstration:")
print("=" * 70)
print()
print("Scenario: Predicting carrier mobility from CVD temperature.")
print()

# Complete evidence (PSP chain)
complete_paths = find_psp_paths(
    kg, start_keywords=["temperature", "CVD"], end_keywords=["mobility"], max_hops=4
)
complete_score = causal_completeness_score(kg, complete_paths, "Predict carrier mobility from CVD temperature")
complete_missing = identify_missing_layers(kg, complete_paths, "Predict carrier mobility from CVD temperature")

print("COMPLETE EVIDENCE (PSP chain):")
for path in complete_paths:
    layers = psp_layers_covered(path, kg)
    print(f"  {' -> '.join(path)}")
    print(f"    Layers: {layers}")
print(f"  Completeness score: {complete_score:.2f}")
print(f"  Missing layers: {[l.value for l in complete_missing]}")
print(f"  -> All three PSP layers covered. Evidence is causally complete.")
print(f"  -> ARIA activates Tier 1 with high confidence.")
print()

# Incomplete evidence (only P -> Pr shortcut)
print("INCOMPLETE EVIDENCE (P -> Pr shortcut only):")
print("  CVD temperature 750C -> higher carrier mobility  (hypothetical shortcut)")
print("  Layers covered: {Processing, Property}")
print("  Completeness score: 0.67")
print("  Missing layers: {Structure}")
print("  -> The Structure layer is missing. The LLM cannot explain WHY")
print("     temperature affects mobility -- it just knows it does.")
print("  -> Naive retrieval would include this edge, potentially misleading")
print("     the LLM into making a confident but unjustified prediction.")
print()

# No evidence
print("NO EVIDENCE (Tier 3 fallback):")
print("  No KG paths found.")
print("  Completeness score: 0.00")
print("  -> The LLM relies on parametric knowledge only.")
print("  -> Predictions may be vague but not actively misleading.")
print()
print("KEY INSIGHT: Incomplete evidence (0.67 completeness) can be WORSE")
print("than no evidence (0.00) because it creates a false sense of support.")
print("ARIA's tier system prevents this by gating evidence activation."

## 5. Evidence Ranking

ARIA ranks retrieved paths by a composite score that combines:

- **Confidence** (weight 0.35): Average edge confidence along the path
- **Evidence richness** (weight 0.30): Fraction of edges with mechanism text
- **PSP coverage** (weight 0.35): Fraction of PSP layers the path traverses

In [ ]:
# Rank paths by evidence quality
paths = find_psp_paths(
    kg, start_keywords=["temperature", "CVD", "doping"], end_keywords=["mobility"], max_hops=4
)

ranked = rank_paths_by_evidence(paths, kg)

print("Evidence Ranking:")
print("=" * 70)
print()
for i, (path, score) in enumerate(ranked, 1):
    details = path_score_details(path, kg)
    layers = psp_layers_covered(path, kg)
    print(f"Rank {i} (score={score:.3f}):")
    print(f"  Path:      {' -> '.join(path)}")
    print(f"  Layers:    {layers}")
    print(f"  Confidence:     {details['confidence']:.3f}")
    print(f"  Evidence rich:  {details['evidence_richness']:.3f}")
    print(f"  PSP coverage:   {details['psp_coverage']:.3f}")
    print()

## 6. Evaluation Metrics

ARIA provides a comprehensive evaluation framework through `MetricsComputer`, which computes:

1. **Causal coherence** (0-1): Intervention consistency, counterfactual reasoning, mechanistic modularity, PSP chain validity
2. **Source grounding** (0-1): KG verification, literature entailment, multi-hop reasoning, source diversity
3. **Internal validity** (0-1): Sufficiency, comprehensiveness, logical consistency, mechanistic precision, forward simulation
4. **PSP consistency** (0-1): Whether the output contains a complete PSP chain

These metrics are grounded in the causal reasoning literature (Pearl 2019, Scholkopf et al. 2021) and the evidence verification literature (Thorne et al. 2018, Wiegreffe & Marasovic 2021).

In [ ]:
from aria.evaluation.metrics import MetricsComputer

# Create a MetricsComputer
metrics = MetricsComputer(kg=kg)

# Example prediction and ground truth
prediction = {
    "predicted_properties": {
        "carrier_type": "n-type",
        "mobility": "50 cm2/Vs",
        "conductivity": "high",
    },
    "processing_conditions": {
        "method": "CVD",
        "temperature": "750C",
    },
    "structure": {
        "crystallinity": "improved",
        "defect_density": "reduced",
    },
    "mechanistic_explanation": {
        "primary_mechanism": "High CVD temperature promotes crystallinity and reduces defects",
        "chain_of_thought": [
            "CVD temperature 750C increases crystallinity",
            "Higher crystallinity reduces scattering",
            "Reduced scattering increases carrier mobility",
        ],
        "quantitative_estimates": {"mobility": "45-55 cm2/Vs"},
        "alternative_mechanisms": "Different temperature could lead to different defect profiles",
    },
    "kg_paths_used": 2,
    "confidence": 0.85,
}

ground_truth = {
    "carrier_type": "n-type",
    "mobility": "50 cm2/Vs",
}

# Note: Full metrics require spaCy and sentence-transformers
# Some sub-metrics may not be available without these dependencies
try:
    psp_consistency = metrics.psp_consistency(prediction)
    print(f"PSP Consistency Score: {psp_consistency:.3f}")
    print()
    print("This score measures whether the prediction contains a complete")
    print("Processing -> Structure -> Property causal chain.")
except Exception as e:
    print(f"Metrics computation requires additional dependencies: {e}")
    print()
    print("Expected PSP consistency score: ~0.7-1.0")
    print("(high because the prediction includes all three PSP components)")

## 7. Benchmark Runner

The `BenchmarkRunner` class orchestrates evaluation across multiple engine modes and tasks. It loads JSONL benchmark files, runs each mode, and computes metrics.

In [ ]:
from aria.evaluation.benchmark import BenchmarkRunner
from aria.types import EngineMode

# Create a benchmark runner
# NOTE: Running benchmarks requires an active Ollama server
runner = BenchmarkRunner(
    kg=kg,
    models=["qwen2:7b"],
    modes=[EngineMode.BASELINE, EngineMode.NAIVE_KG, EngineMode.ARIA],
)

print("BenchmarkRunner initialized.")
print(f"  Models: {runner.models}")
print(f"  Modes: {[m.value for m in runner.modes]}")
print()
print("To run benchmarks, create a JSONL task file and call:")
print("  runner.run(task_file='benchmark_tasks.jsonl', output_dir='results/')")
print()
print("Task file format (JSONL, one task per line):")
task_example = {
    "id": "task_001",
    "query": {
        "material": "MoS2",
        "processing": {"method": "CVD", "temperature": "750C"},
        "target_property": "carrier mobility",
    },
    "ground_truth": {
        "carrier_type": "n-type",
        "mobility": "50 cm2/Vs",
    },
}
print(json.dumps(task_example, indent=2))

## 8. LLM-as-Judge Evaluation

ARIA includes a domain-specific LLM-as-judge evaluation system (`LLMJudge`) that scores predictions on four rubrics:

| Rubric | Max Score | Description |
|--------|-----------|-------------|
| Processing Feasibility | 40 | Thermodynamic and kinetic viability of synthesis conditions |
| Structure Emergence | 30 | Accuracy of predicted structural outcomes |
| Property Consistency | 20 | Coherence between properties and structure/processing |
| Causal PSP Reasoning | 10 | Quality of P->S->P causal chain |
| **Total** | **100** | |

The judge evaluates each rubric using detailed scoring guidelines specific to 2D materials science.

In [ ]:
from aria.evaluation.judge import LLMJudge, METRICS

# Display the evaluation rubrics
print("LLM-as-Judge Evaluation Rubrics:")
print("=" * 70)
print()
for key, metric in METRICS.items():
    print(f"{metric['name']} ({metric['max_score']} points)")
    print(f"  {metric['description']}")
    print()

# Create a judge instance (requires Ollama or OpenAI)
print("To use the judge:")
print("  judge = LLMJudge(backend='ollama', model='qwen2:7b')")
print("  result = judge.evaluate_all_metrics(query, prediction, ground_truth)")
print()
print("The judge returns:")
print("  - metric_scores: Individual scores with justifications")
print("  - overall_score: Sum of all rubric scores (0-100)")
print("  - overall_score_normalized: Percentage (0-100%)")

## 9. Visualizing Causal Traces

ARIA provides visualization functions that render causal traces as PSP chain diagrams.

In [ ]:
# Visualize the causal trace
plot_causal_trace(result_with_trace, output_path="causal_trace_evaluation.png")
print("Causal trace visualization saved.")
print()
print("The visualization shows:")
print("  - Red (Processing) nodes: Synthesis conditions")
print("  - Blue (Structure) nodes: Structural changes")
print("  - Green (Property) nodes: Resulting properties")
print("  - Edge annotations: Confidence scores")
print("  - Title: Tier and overall confidence")

In [ ]:
# Compare results across tiers
tier_results = [
    ARIAResult(
        answer={"mobility": "30-60 cm2/Vs"},
        tier=ReasoningTier.FALLBACK, confidence=0.5,
        reasoning_type="baseline_llm", causal_trace=[],
        kg_paths_used=0, mode="baseline", model="qwen2:7b",
    ),
    ARIAResult(
        answer={"mobility": "40-70 cm2/Vs"},
        tier=ReasoningTier.FALLBACK, confidence=0.6,
        reasoning_type="naive_kg", causal_trace=[],
        kg_paths_used=2, mode="naive_kg", model="qwen2:7b",
    ),
    ARIAResult(
        answer={"mobility": "50 cm2/Vs"},
        tier=ReasoningTier.DIRECT, confidence=0.85,
        reasoning_type="direct_path",
        causal_trace=result_with_trace.causal_trace,
        kg_paths_used=2, mode="aria", model="qwen2:7b",
    ),
]

plot_tier_comparison(tier_results, output_path="tier_comparison_evaluation.png")
print("Tier comparison visualization saved.")

## 10. Chain-of-Thought Transparency (ARIA_FULL mode)

In `aria_full` mode, ARIA provides full chain-of-thought transparency through the `ChainOfThought` and `ReasoningStep` types. Each step includes source attribution and confidence scores.

In [ ]:
# Demonstrate the ChainOfThought structure
cot_example = ChainOfThought(
    query_context={"type": "forward", "material": "MoS2", "method": "CVD"},
    reasoning_steps=[
        ReasoningStep(
            step_id="kg_retrieval",
            description="Found 2 paths at tier 1 (direct match)",
            evidence_sources=[
                KnowledgeSource(
                    source_id="kg_node_1",
                    content="CVD temperature 750C",
                    source_type="kg_node",
                    confidence=1.0,
                    context="Processing node",
                ),
                KnowledgeSource(
                    source_id="kg_mech_1",
                    content="High CVD temperature promotes ordered crystal growth",
                    source_type="kg_mechanism",
                    confidence=0.95,
                    context="Mechanism for temperature -> crystallinity",
                ),
            ],
            reasoning_type="retrieval",
            confidence=1.0,
            intermediate_conclusion="Tier 1 selected",
        ),
        ReasoningStep(
            step_id="llm_synthesis",
            description="Generated prediction via Tier 1",
            evidence_sources=[],
            reasoning_type="synthesis",
            confidence=0.85,
            intermediate_conclusion="Predicted n-type mobility ~50 cm2/Vs",
        ),
    ],
    final_reasoning="CVD at 750C improves crystallinity and reduces defects,",
    final_result={"carrier_type": "n-type", "mobility": "50 cm2/Vs"},
    confidence_breakdown={"kg_retrieval": 1.0, "llm_synthesis": 0.85},
    source_attribution={"kg_sources": 2, "literature_sources": 0, "total_sources": 2},
    tier=1,
    kg_paths_used=2,
    literature_papers_used=0,
)

print("Chain-of-Thought Structure (aria_full mode):")
print("=" * 60)
print(f"  Query context: {cot_example.query_context}")
print(f"  Tier: {cot_example.tier}")
print(f"  KG paths used: {cot_example.kg_paths_used}")
print(f"  Literature papers: {cot_example.literature_papers_used}")
print(f"  Source attribution: {cot_example.source_attribution}")
print()
for step in cot_example.reasoning_steps:
    print(f"  Step: {step.step_id}")
    print(f"    Type: {step.reasoning_type}")
    print(f"    Description: {step.description}")
    print(f"    Confidence: {step.confidence}")
    print(f"    Conclusion: {step.intermediate_conclusion}")
    print(f"    Sources: {len(step.evidence_sources)}")
    print()

## 11. Summary and Key Takeaways

In this tutorial, we:

1. **Analyzed causal traces** in ARIAResult -- understanding how ARIA explains its reasoning through PSP chains
2. **Computed causal completeness scores** -- quantifying whether evidence covers all required PSP layers
3. **Demonstrated contextual tunneling** -- how incomplete evidence can be worse than no evidence
4. **Ranked evidence** by composite quality scores
5. **Explored evaluation metrics** -- causal coherence, source grounding, internal validity, PSP consistency
6. **Learned about the LLM-as-judge** system for domain-specific evaluation
7. **Visualized causal traces** with PSP layer coloring
8. **Examined chain-of-thought transparency** in aria_full mode

### The Central Finding

> **Retrieval without causal completeness can harm reasoning.** ARIA's evaluation framework quantifies this: predictions based on incomplete PSP chains score lower on causal coherence, source grounding, and PSP consistency. The causal completeness metric C(E, q) provides a principled way to gate evidence activation.

This concludes the tutorial series. For more details, see the API documentation at `docs/api.md`.